In [1]:
import uproot

path = "/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS/ICARUS_CC0pi_GUNDAM/data/Fitter/dpT/Asimov_StatSyst/toy_0001.root"

f = uproot.open(path)

def walk(obj, prefix=""):
    """Recursively print directory/tree structure."""
    for key in obj.keys(recursive=False):
        full = f"{prefix}/{key}" if prefix else key
        item = obj[key]
        cls = type(item).__name__
        print(f"{full}  [{cls}]")
        # Recurse into subdirectories
        if hasattr(item, "keys") and not hasattr(item, "num_entries"):
            walk(item, full)
        # If it's a TTree, list its branches
        elif hasattr(item, "num_entries"):
            print(f"    -> TTree with {item.num_entries} entries")
            for branch in item.keys():
                print(f"       {branch}")

walk(f)

gundam;1  [ReadOnlyDirectory]
gundam;1/runtime;1  [ReadOnlyDirectory]
gundam;1/runtime;1/date_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/user_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/host_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/pwd_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/os_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/dist_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/arch_TNamed;1  [Model_TNamed]
gundam;1/runtime;1/commandLine_TNamed;1  [Model_TNamed]
gundam;1/build;1  [ReadOnlyDirectory]
gundam;1/build;1/version_TNamed;1  [Model_TNamed]
gundam;1/build;1/root;1  [ReadOnlyDirectory]
gundam;1/build;1/root;1/version_TNamed;1  [Model_TNamed]
gundam;1/build;1/root;1/date_TNamed;1  [Model_TNamed]
gundam;1/build;1/root;1/install_TNamed;1  [Model_TNamed]
gundam;1/config;1  [ReadOnlyDirectory]
gundam;1/config;1/unfoldedJson_TNamed;1  [Model_TNamed]
FitterEngine;1  [ReadOnlyDirectory]
FitterEngine;1/preFit;1  [ReadOnlyDirectory]
FitterEngine;1/preFit;1/parameters;1  [ReadOnlyDirectory]

In [6]:
import uproot, pandas as pd, glob

PROD = '/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS/ICARUS_CC0pi_GUNDAM/data/Fitter/dpT/Asimov_StatSyst'

files = sorted(glob.glob(f"{PROD}/toy_*.root"))
print(f"Found {len(files)} files")
assert files, "No toy files found."

rows = []
for path in files:
    try:
        f = uproot.open(path)
        a = f["FitterEngine/postFit/bestFitStats"].arrays(library="np")
        rows.append({
            "seed":          int(a["toyIndex"][0]),
            "converged":     bool(a["fitConverged"][0]),
            "fit_status":    int(a["fitStatusCode"][0]),
            "cov_status":    int(a["covStatusCode"][0]),
            "edm":           float(a["edmBestFit"][0]),
            "ndof":          int(a["nbDegreeOfFreedom"][0]),
            "llh_total":     float(a["totalLikelihoodAtBestFit"][0]),
            "llh_stat":      float(a["statLikelihoodAtBestFit"][0]),
            "llh_penalty":   float(a["penaltyLikelihoodAtBestFit"][0]),
            "llh_stat_sig":  float(a["selection_reco_dpT"][0][0]),
            "llh_stat_side": float(a["sideband_reco_dpT"][0][0]),
            "llh_pen_tmpl":  float(a["Template_Parameter_true_dpT_lp_genie"][0][0]),
            "llh_pen_xsec":  float(a["Cross_section_Systematics"][0][0]),
            "llh_pen_flux":  float(a["Multisigma_Flux_Systematics"][0][0]),
            "llh_pen_det":   float(a["Detector_Systematics"][0][0]),
        })
    except Exception as e:
        print(f"FAILED {path}: {e}")

df = pd.DataFrame(rows)
print(df)
df.to_csv("toys_summary.csv", index=False)
print(f"{len(df)} toys; {(~df['converged']).sum()} non-converged")

Found 1 files
   seed  converged  fit_status  cov_status           edm  ndof  llh_total  \
0     1       True           0           3  5.380131e-08    19   7.141473   

   llh_stat  llh_penalty  llh_stat_sig  llh_stat_side  llh_pen_tmpl  \
0   1.64763     5.493844      0.733694       0.913936           0.0   

   llh_pen_xsec  llh_pen_flux  llh_pen_det  
0      4.332651      0.493503      0.66769  
1 toys; 0 non-converged


In [7]:
"""
Plot LLH distributions from the stat+syst toy ensemble.
Follows Howard's layout: histograms with chi^2 overlay; data fit value
to be added as a red dashed line once available.

Run after copying toys_summary.csv from the GPVM.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

CSV   = "/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS/ICARUS_CC0pi_GUNDAM/data/Fitter/dpT/Asimov_StatSyst/toys_summary.csv"   # adapt path
OUT   = "."                  # output directory for PNGs

df = pd.read_csv(CSV)
print(f"Loaded {len(df)} toys")
print(f"  Non-converged:  {(~df['converged']).sum()}")
print(f"  Imperfect Hess: {(df['cov_status'] != 3).sum()}")

# Use only converged toys for the LLH plots
ok = df[df['converged']].copy()
print(f"Using {len(ok)} converged toys for plots")

# k for the chi^2 overlay (ndof from the fit itself)
k_total = int(ok['ndof'].iloc[0])
print(f"chi^2 reference k = {k_total}")


def plot_llh(values, k_chi2, xlabel, title, outfile, nbins=40):
    """Histogram of LLH values with chi^2_k overlay."""
    fig, ax = plt.subplots(figsize=(6, 4.5))

    counts, edges, _ = ax.hist(
        values, bins=nbins, range=(0, max(values.max(), k_chi2 * 2.5)),
        histtype='stepfilled', alpha=0.4, color='steelblue',
        edgecolor='steelblue', linewidth=1.5,
        label=f'Toy data fits ({len(values)})'
    )

    # chi^2_k overlay, normalized to the histogram's integral
    x = np.linspace(0, edges[-1], 400)
    pdf = chi2.pdf(x, df=k_chi2)
    binwidth = edges[1] - edges[0]
    ax.plot(x, pdf * len(values) * binwidth,
            'k--', linewidth=1.5, label=rf'$\chi^2_{{k={k_chi2}}}$')

    # Empirical summary stats
    mean = values.mean()
    std  = values.std()
    ax.text(0.97, 0.97,
            f"mean = {mean:.2f}\nstd  = {std:.2f}\nexpected mean = {k_chi2}",
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of toys")
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(f"{OUT}/{outfile}", dpi=150)
    plt.close(fig)
    print(f"  -> {outfile}")


# ----- LLH_total -----
plot_llh(ok['llh_total'], k_total,
         r"$\mathrm{LLH}_{\mathrm{total}}$",
         r"Total LLH — ICARUS NuMI $1\mu Np0\pi^\pm$, $\delta p_T$",
         "llh_total.png")

# ----- LLH_stat split by sample -----
# Howard uses a chi^2 with k = (bins in sample) as a rough reference;
# 13 bins each for signal and sideband (26 total).
plot_llh(ok['llh_stat_sig'], 13,
         r"$\mathrm{LLH}_{\mathrm{stat}}$",
         "Stat LLH — Signal selection",
         "llh_stat_signal.png")

plot_llh(ok['llh_stat_side'], 13,
         r"$\mathrm{LLH}_{\mathrm{stat}}$",
         "Stat LLH — Sideband",
         "llh_stat_sideband.png")

# ----- LLH_penalty split by parameter group -----
# Reference k = nominal number of constrained params in each group
# (actual effective DOF will differ due to PCA / correlations)
plot_llh(ok['llh_pen_xsec'], 51,
         r"$\mathrm{LLH}_{\mathrm{penalty}}$",
         "Penalty LLH — Cross-section systematics",
         "llh_penalty_xsec.png")

plot_llh(ok['llh_pen_flux'], 27,
         r"$\mathrm{LLH}_{\mathrm{penalty}}$",
         "Penalty LLH — Flux systematics",
         "llh_penalty_flux.png")

plot_llh(ok['llh_pen_det'], 9,
         r"$\mathrm{LLH}_{\mathrm{penalty}}$",
         "Penalty LLH — Detector systematics",
         "llh_penalty_det.png")

print("\nDone. Six PNGs written to", OUT)

Loaded 1000 toys
  Non-converged:  0
  Imperfect Hess: 3
Using 1000 converged toys for plots
chi^2 reference k = 19
  -> llh_total.png
  -> llh_stat_signal.png
  -> llh_stat_sideband.png
  -> llh_penalty_xsec.png
  -> llh_penalty_flux.png
  -> llh_penalty_det.png

Done. Six PNGs written to .
